# Model Training and Selection

This notebook compares the four requested classifiers using stratified cross-validation on the training split. Exact duplicates are removed before splitting so a repeated row cannot appear in both partitions. Preprocessing stays inside each sklearn pipeline and is fitted only on training folds.

## Load and Prepare the Dataset

The shared helper loads `data/cm1.csv`, reports exact duplicate removal, and excludes identifier-like fields if present. A stratified holdout is created before any model fitting.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

project_root = next(
    path for path in (Path.cwd(), Path.cwd().parent)
    if (path / "src" / "train_model.py").is_file()
)
sys.path.insert(0, str(project_root))
from src.train_model import RANDOM_STATE, TARGET_COLUMN, build_model_pipelines, load_dataset

features, target, data_summary = load_dataset(project_root / "data" / "cm1.csv")
removed_duplicate_count = data_summary["duplicate_rows_removed"]
identifier_columns = data_summary["identifier_columns"]

print(f"Rows after removing {removed_duplicate_count:,} exact duplicates: {len(features):,}")
print(f"Feature columns after excluding identifiers: {features.shape[1]}")
print("Identifier columns removed:", identifier_columns if identifier_columns else "none")
print("Target distribution:")
display(target.value_counts().rename_axis(TARGET_COLUMN).to_frame("count"))

X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.2,
    stratify=target,
    random_state=RANDOM_STATE,
)
print(f"Training rows: {len(X_train):,} | Holdout rows: {len(X_test):,}")
print("The holdout is reserved for final evaluation and is not used for model selection.")

Rows after removing 56 exact duplicates: 442
Feature columns after excluding identifiers: 21
Identifier columns removed: none
Target distribution:


,count
defects,
False,394
True,48


Training rows: 353 | Holdout rows: 89
The holdout is reserved for final evaluation and is not used for model selection.


## Compare Models with Training-Only Cross-Validation

Each model receives its own complete preprocessing pipeline. Balanced class weights (or XGBoost's training-fold positive-class weight) give the minority defective class more influence. F1 is the selection metric because it balances precision and recall; recall is particularly useful when missing a defective module is costly.

In [2]:
model_pipelines = build_model_pipelines(X_train, y_train)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []

for model_name, pipeline in model_pipelines.items():
    fold_scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        scoring="f1",
        cv=cv,
        n_jobs=1,
    )
    cv_rows.append({
        "Model": model_name,
        "Mean CV F1": fold_scores.mean(),
        "CV F1 Std": fold_scores.std(),
    })

training_comparison = (
    pd.DataFrame(cv_rows)
    .sort_values("Mean CV F1", ascending=False)
    .reset_index(drop=True)
)
display(training_comparison)

selected_model_name = training_comparison.loc[0, "Model"]
selected_pipeline = model_pipelines[selected_model_name]
selected_pipeline.fit(X_train, y_train)
print(f"Selected by highest training-only cross-validation F1: {selected_model_name}")
print("The fitted pipeline includes preprocessing and the classifier.")

,Model,Mean CV F1,CV F1 Std
0,Logistic Regression,0.402569,0.063065
1,XGBoost,0.256702,0.168823
2,Decision Tree,0.178388,0.209220
3,Random Forest,0.000000,0.000000


Selected by highest training-only cross-validation F1: Logistic Regression
The fitted pipeline includes preprocessing and the classifier.
